<a href="https://colab.research.google.com/github/pksX01/PySpark_Tutorials/blob/main/ShareChat_SQL_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.4/281.4 MB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.7/199.7 KB 12.5 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.3.1-py2.py3-none-any.whl size=281845512 sha256=203ce3ea2e03fe1bc4ab452cba578e54e9898578e91725662ef84fd26c3d05b0
  Stored in directory: /root/.cache/pip/wheels/43/dc/11/ec201cd671da62fa9c5cc77078235e40722170ceba231d7598
Successfully built pyspark


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.enableHiveSupport().getOrCreate()

In [ ]:
spark.sql("Create table if not exists Moj_user (Userid int, Signup_time timestamp, Handle string, Date_of_birth date, Gender string)")
spark.sql("Create table if not exists Follow (Userid int, Followerid int, Follow_time timestamp)")
spark.sql("Create table if not exists Unfollow (Userid int, Followerid int, Unfollow_time timestamp)")

DataFrame[]

In [ ]:
spark.sql("show tables").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|  default|   follow|      false|
|  default| moj_user|      false|
|  default| unfollow|      false|
+---------+---------+-----------+



In [ ]:
spark.sql('truncate table Moj_user')

DataFrame[]

In [ ]:
spark.sql("insert into Moj_user values (259235535, cast('2021-01-04 12:43:53' as timestamp), 'ArjunSaluja', cast('2000-09-12' as date), 'M')")
spark.sql("insert into Moj_user values (678493443, cast('2018-09-30 16:32:45' as timestamp), 'NatashaArora', cast('1996-10-23' as date), 'F')")

DataFrame[]

In [ ]:
spark.sql('select * from Moj_user').show()

+---------+-------------------+------------+-------------+------+
|   Userid|        Signup_time|      Handle|Date_of_birth|Gender|
+---------+-------------------+------------+-------------+------+
|259235535|2021-01-04 12:43:53| ArjunSaluja|   2000-09-12|     M|
|678493443|2018-09-30 16:32:45|NatashaArora|   1996-10-23|     F|
+---------+-------------------+------------+-------------+------+



In [ ]:
spark.sql("insert into Follow values (259235535, cast(5324523442 as int), cast('2022-02-06 19:03:49' as timestamp))")
spark.sql("insert into Follow values (259235535, cast(4234235234 as int), cast('2021-11-09 23:11:19' as timestamp))")

DataFrame[]

In [ ]:
spark.sql('select * from Follow').show()

+---------+----------+-------------------+
|   Userid|Followerid|        Follow_time|
+---------+----------+-------------------+
|259235535| -60732062|2021-11-09 23:11:19|
|259235535|1029556146|2022-02-06 19:03:49|
+---------+----------+-------------------+



In [ ]:
spark.sql("insert into Unfollow values (259235535, cast(5324523442 as int), cast('2022-03-04 21:23:43' as timestamp))")
spark.sql("insert into Unfollow values (259235535, cast(2342342342 as int), cast('2021-12-21 17:20:22' as timestamp))")

DataFrame[]

In [ ]:
spark.sql('select * from Unfollow').show()

+---------+-----------+-------------------+
|   Userid| Followerid|      Unfollow_time|
+---------+-----------+-------------------+
|259235535| 1029556146|2022-03-04 21:23:43|
|259235535|-1952624954|2021-12-21 17:20:22|
+---------+-----------+-------------------+



In [ ]:
spark.sql("""
  select * from
  moj_user a
  inner join follow b on a.Userid = b.Userid
  inner join unfollow c on a.Userid = c.Userid
  where a.Gender = 'M'and b.Followerid not in (select Followerid from unfollow)
  or (b.Followerid in (select Followerid from unfollow) and b.Follow_time > c.Unfollow_time)
""").show()

+---------+-------------------+-----------+-------------+------+---------+----------+-------------------+---------+-----------+-------------------+
|   Userid|        Signup_time|     Handle|Date_of_birth|Gender|   Userid|Followerid|        Follow_time|   Userid| Followerid|      Unfollow_time|
+---------+-------------------+-----------+-------------+------+---------+----------+-------------------+---------+-----------+-------------------+
|259235535|2021-01-04 12:43:53|ArjunSaluja|   2000-09-12|     M|259235535|1029556146|2022-02-06 19:03:49|259235535|-1952624954|2021-12-21 17:20:22|
|259235535|2021-01-04 12:43:53|ArjunSaluja|   2000-09-12|     M|259235535| -60732062|2021-11-09 23:11:19|259235535|-1952624954|2021-12-21 17:20:22|
|259235535|2021-01-04 12:43:53|ArjunSaluja|   2000-09-12|     M|259235535| -60732062|2021-11-09 23:11:19|259235535| 1029556146|2022-03-04 21:23:43|
+---------+-------------------+-----------+-------------+------+---------+----------+-------------------+-------

In [ ]:
spark.sql("""
  select * from follow where Followerid not in (select Followerid from unfollow)
  --or (Followerid in (select Followerid from unfollow) and follow_time > (select unfollow_time from unfollow))
""").show()

+---------+----------+-------------------+
|   Userid|Followerid|        Follow_time|
+---------+----------+-------------------+
|259235535| -60732062|2021-11-09 23:11:19|
+---------+----------+-------------------+



In [ ]:
spark.sql("""
  select * from follow a inner join unfollow b on a.Userid = b.Userid and a.Followerid = b.Followerid and a.Follow_time > b.Unfollow_time
""").show()

+------+----------+-----------+------+----------+-------------+
|Userid|Followerid|Follow_time|Userid|Followerid|Unfollow_time|
+------+----------+-----------+------+----------+-------------+
+------+----------+-----------+------+----------+-------------+



In [ ]:
spark.sql("""
  select count(*) from
  (select count(distinct b.Followerid) as cnt, a.Userid from
    moj_user a
    inner join follow b on a.Userid = b.Userid
    inner join unfollow c on a.Userid = c.Userid
    where a.Gender = 'M'and b.Followerid not in (select Followerid from unfollow)
    or (b.Followerid = c.Followerid and b.Follow_time > c.Unfollow_time)
    group by a.Userid
  ) tbl where cnt >= 1
""").show()

+--------+
|count(1)|
+--------+
|       1|
+--------+



In [ ]:
spark.sql("""
  select count(distinct b.Followerid) as cnt, a.Userid from
    moj_user a
    inner join follow b on a.Userid = b.Userid
    inner join unfollow c on a.Userid = c.Userid
    where a.Gender = 'M'and b.Followerid not in (select Followerid from unfollow)
    or (b.Followerid = c.Followerid and b.Follow_time > c.Unfollow_time)
    group by a.Userid
""").show()

+---+---------+
|cnt|   Userid|
+---+---------+
|  1|259235535|
+---+---------+



In [ ]:
spark.sql("""
  select count(*) from
  (select count(distinct b.Followerid) as cnt, a.Userid from
    moj_user a
    inner join follow b on a.Userid = b.Userid
    inner join unfollow c on a.Userid = c.Userid
    where a.Gender = 'F'and b.Followerid not in (select Followerid from unfollow)
    or (b.Followerid = c.Followerid and b.Follow_time > c.Unfollow_time)
    group by a.Userid
  ) tbl where cnt > 10000
""").show()

+--------+
|count(1)|
+--------+
|       0|
+--------+



In [ ]:
spark.sql("""
  select count(*),  Userid from
  (select count(distinct b.Followerid) as cnt, a.Userid from
    moj_user a
    inner join follow b on a.Userid = b.Userid
    inner join unfollow c on a.Userid = c.Userid
    where a.Gender = 'M'and b.Followerid not in (select Followerid from unfollow)
    or (b.Followerid = c.Followerid and b.Follow_time > c.Unfollow_time)
    group by a.Userid
  ) tbl where cnt >= 1 group by Userid order by
""").show()

+--------+---------+
|count(1)|   Userid|
+--------+---------+
|       1|259235535|
+--------+---------+



In [ ]:
spark.sql("""
  select count(distinct b.Followerid) as cnt, a.Userid from
    moj_user a
    inner join follow b on a.Userid = b.Userid
    inner join unfollow c on a.Userid = c.Userid
    where a.Gender = 'M'and b.Followerid not in (select Followerid from unfollow)
    or (b.Followerid = c.Followerid and b.Follow_time > c.Unfollow_time)
    group by a.Userid having count(distinct b.Followerid) >= 1 order by cnt desc limit 25
  """).show()

+---+---------+
|cnt|   Userid|
+---+---------+
|  1|259235535|
+---+---------+



In [ ]:
spark.sql("""
  select count(distinct b.Followerid) as cnt, a.Userid from
    moj_user a
    inner join follow b on a.Userid = b.Userid
    inner join unfollow c on a.Userid = c.Userid
    where a.Gender = 'F'and b.Followerid not in (select Followerid from unfollow)
    or (b.Followerid = c.Followerid and b.Follow_time > c.Unfollow_time)
    group by a.Userid having count(distinct b.Followerid) >= 10000 order by cnt desc limit 25
  """).show()

+---+------+
|cnt|Userid|
+---+------+
+---+------+



In [ ]:
spark.sql("create table dummy (col1 string)")

DataFrame[]

In [ ]:
spark.sql("insert into dummy values ('24556')")
spark.sql("insert into dummy values ('76678')")
spark.sql("insert into dummy values ('')")
spark.sql("insert into dummy values (NULL)")

DataFrame[]

In [ ]:
spark.sql("select * from dummy").show()

+-----+
| col1|
+-----+
|     |
|76678|
| null|
|24556|
+-----+



In [ ]:
spark.sql("select * from dummy where col1 is not null or col1 <> ''").show()

+-----+
| col1|
+-----+
|     |
|76678|
|24556|
+-----+



In [ ]:
spark.sql("select * from dummy where col1 is not null and col1 <> ''").show()

+-----+
| col1|
+-----+
|76678|
|24556|
+-----+

